# 03 — MuJoCoのGo2 Plant

制御器の外側にある、状態を生成しトルクを受け取る物理系を確認します。

**前提**: `02_vectors_units_and_frames.ipynb`

> 読み方: 「直感 → 数式 → 上流コード → 小実験 → 解釈」の順です。
> `実装事実` と書いた箇所は現行 `external/Quadruped-PyMPC` のコード、
> `学習用モデル` は理解のために単純化した再実装です。

In [1]:
from pathlib import Path
import os, sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebook_pympc":
    ROOT = ROOT.parent
PYMPC_ROOT = ROOT / "external" / "Quadruped-PyMPC"
assert PYMPC_ROOT.exists(), f"Quadruped-PyMPC が見つかりません: {PYMPC_ROOT}"
if str(PYMPC_ROOT) not in sys.path:
    sys.path.insert(0, str(PYMPC_ROOT))

os.environ.setdefault("ACADOS_SOURCE_DIR", str(PYMPC_ROOT / "quadruped_pympc" / "acados"))
os.environ.setdefault("MUJOCO_GL", "egl")
print("workspace :", ROOT)
print("PyMPC root:", PYMPC_ROOT)

workspace : /home/takuya/work/mpc_dog
PyMPC root: /home/takuya/work/mpc_dog/external/Quadruped-PyMPC


## 完全モデルと縮約モデル

MuJoCoは浮遊基部と12関節を持つ全身モデルを積分します。
一方MPCは胴体を単一剛体として近似します。

\[
q\in\mathbb{R}^{19},\quad \dot q\in\mathbb{R}^{18},\quad
\tau\in\mathbb{R}^{12}
\]

quaternionが4要素なので、浮遊基部の `nq` と `nv` は1だけ異なります。

In [2]:
import numpy as np
from gym_quadruped.quadruped_env import QuadrupedEnv
from quadruped_pympc import config as cfg

env = QuadrupedEnv(
    robot=cfg.robot, scene="flat", sim_dt=cfg.simulation_params["dt"],
    ref_base_lin_vel=0.0, ref_base_ang_vel=0.0,
    ground_friction_coeff=0.8, base_vel_command_type="forward",
    state_obs_names=(),
)
env.reset(random=False)
print("nq, nv, nu:", env.mjModel.nq, env.mjModel.nv, env.mjModel.nu)
print("qpos shape:", env.mjData.qpos.shape)
print("qvel shape:", env.mjData.qvel.shape)
assert (env.mjModel.nq, env.mjModel.nv, env.mjModel.nu) == (19, 18, 12)
env.close()

nq, nv, nu: 19 18 12
qpos shape: (19,)
qvel shape: (18,)


/home/takuya/work/mpc_dog/.venv/lib/python3.11/site-packages/gymnasium/spaces/box.py:231: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/home/takuya/work/mpc_dog/.venv/lib/python3.11/site-packages/gymnasium/spaces/box.py:297: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


## インターフェース

`simulation.py` はworld座標の足位置・COM速度、base座標の角速度、
Jacobian、質量行列などを集めて `compute_actions` に渡します。
戻った脚別トルクをactuator順へ詰め、上限の90%でclipして `env.step(action)` します。

MPCの予測が正しくても、最終トルクclipが頻発すれば実機Plantは予測通り動きません。
したがって後の診断では「目標GRF」「変換後トルク」「clip後トルク」を分けます。

## 章末チェック

出力を眺めるだけでなく、次を自分の言葉で答えてください。

1. この章の入力・出力の shape、単位、座標系は何か。
2. 変更可能な量と、他の章から渡される量は何か。
3. パラメータを2倍にしたとき、どのグラフがどちらへ変化するか。
4. 現行実装の事実と、学習用の近似を区別できるか。